This notebook runs the full multi-asset experiment suite (Experiments A through E) across 10 bank stocks — JPM, C, WFC, GS, MS, PNC, USB, FITB, MTB, and BAC. It imports and calls the existing pipeline from `run_multiasset_experiments.py` and `scripts/`, which must be on the Python path. All trained weights are cached under `results_multiasset/_cache/`; re-running any cell skips training for experiments that have already completed. Final results, NLL tables, and a markdown summary are written to `results_multiasset/`.

In [ ]:
import sys
import torch
import numpy as np

if torch.cuda.is_available():
    device_name = "cuda"
elif torch.backends.mps.is_available():
    device_name = "mps"
else:
    device_name = "cpu"

print(f"Device           : {device_name}")
print(f"torch version    : {torch.__version__}")
print(f"numpy version    : {np.__version__}")
print(f"Python version   : {sys.version.split()[0]}")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
import sys
from pathlib import Path

project_root = Path(".").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from scripts.config import make_config
from run_multiasset_experiments import run_all_experiments, RESULTS_DIR, TARGET_STOCKS

cfg = make_config()
print(f"horizons  : {cfg.horizons}")
print(f"seeds     : {cfg.seeds}")
print(f"swa_epochs: {cfg.swa_epochs}")
print(f"max_epochs: {cfg.max_epochs}  patience: {cfg.patience}")

In [ ]:
import contextlib
import sys
from pathlib import Path

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
log_path = RESULTS_DIR / "run.log"

class _Tee:
    def __init__(self, *streams):
        self._streams = streams
    def write(self, data):
        for s in self._streams:
            s.write(data)
            s.flush()
    def flush(self):
        for s in self._streams:
            s.flush()

with open(log_path, "w") as _log_file:
    with contextlib.redirect_stdout(_Tee(sys.stdout, _log_file)):
        run_all_experiments(cfg)

print(f"\nLog written to {log_path}")

In [ ]:
from pathlib import Path

RESULTS_DIR_PATH = Path("results_multiasset")

summary_path = RESULTS_DIR_PATH / "summary.md"
assert summary_path.exists(), f"summary.md not found at {summary_path}"
print(f"summary.md found: {summary_path}")

missing = []
for ticker in TARGET_STOCKS:
    for subdir in ["baseline", "swa_bestsigma"]:
        p = RESULTS_DIR_PATH / ticker / subdir
        if not p.exists():
            missing.append(str(p))

if missing:
    print("MISSING output directories:")
    for m in missing:
        print(f"  {m}")
else:
    print("All 20 expected ticker/experiment directories present.")

weight_files = list(RESULTS_DIR_PATH.rglob("weights/*.pt"))
print(f"Total .pt weight files saved: {len(weight_files)}")

failed_markers = list(RESULTS_DIR_PATH.rglob("FAILED"))
if failed_markers:
    print(f"\nWARNING: {len(failed_markers)} FAILED marker(s) found:")
    for f in failed_markers:
        print(f"  {f}")
else:
    print("No FAILED markers — all experiments completed successfully.")

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(summary_path.read_text())